In [1]:
import sys
sys.path.append('..')

import os
import re
import glob
import pandas as pd

from nnspike.utils import extract_video_frames
from nnspike.data import create_label_dataframe, sort_by_frames_number, label_dataset_by_opencv, label_dataset_by_model, augment_dataset, set_spike_status
from nnspike.constants import  ROI_CNN

course = "right" # "right" or "left"

## Extract Frames from Videos

In [2]:
def get_all_avi_files(directory_path="C:/Users/MSAD/github/nnspike/storage/20250902/videos/", filter_timestamp=None):
    """
    Get all AVI files from the specified directory with their timestamps.
    
    Args:
        directory_path (str): Path to the directory containing AVI files
        filter_timestamp (str, optional): If set, only return files matching this timestamp pattern (supports wildcards with *)
    
    Returns:
        list: List of tuples containing (file_path, timestamp)
    """
    import os
    import glob
    import re
    import fnmatch

    # Use glob to find all .avi files in the directory
    avi_files = glob.glob(os.path.join(directory_path, "*.avi"))
    avi_files = [path.replace("\\", "/") for path in avi_files]
    
    # Sort the files for consistent ordering
    avi_files.sort()
    
    # Extract timestamps and create tuples
    result = []
    for avi_file in avi_files:
        # Extract filename without extension
        filename = os.path.basename(avi_file)
        filename_no_ext = os.path.splitext(filename)[0]
        
        # Extract timestamp from filename (assuming format: timestamp_picamera.avi)
        # This will extract the part before '_picamera'
        timestamp_match = re.match(r'^(\d{14})_.*', filename_no_ext)
        if timestamp_match:
            timestamp = timestamp_match.group(1)
        else:
            # If timestamp pattern not found, use the full filename without extension
            timestamp = filename_no_ext

        # If filter_timestamp is set, only include matching files using pattern matching
        if filter_timestamp is None or fnmatch.fnmatch(timestamp, filter_timestamp):
            result.append((avi_file, timestamp))
    
    return result

def extract_frames_from_avi_files(avi_files_with_timestamps, base_output_dir="C:/Users/MSAD/github/nnspike/storage/20250902/frames/"):
    """
    Extract frames from all AVI files and save them to folders named by timestamp.
    
    Args:
        avi_files_with_timestamps (list): List of tuples containing (file_path, timestamp)
        base_output_dir (str): Base directory where frame folders will be created
    
    Returns:
        list: List of tuples containing (output_directory, timestamp)
    """
    output_directories_with_timestamps = []
    
    for avi_file, timestamp in avi_files_with_timestamps:
        # Create output directory path
        output_dir = os.path.join(base_output_dir, timestamp)
        
        # Create directory if it doesn't exist
        os.makedirs(output_dir, exist_ok=True)
        
        # Add trailing slash for extract_video_frames function
        output_dir_with_slash = output_dir + "/"
        
        filename = os.path.basename(avi_file)
        print(f"Extracting frames from {filename} to {output_dir_with_slash}")
        
        try:
            # Extract frames using the nnspike utility function
            extract_video_frames(avi_file, output_dir_with_slash)
            print(f"✓ Successfully extracted frames to {timestamp}/")
            # Add successful output directory and timestamp to the list
            output_directories_with_timestamps.append((output_dir, timestamp))
        except Exception as e:
            print(f"✗ Error extracting frames from {filename}: {str(e)}")
    
    return output_directories_with_timestamps



In [3]:
# Get all AVI files with their timestamps
avi_files_with_timestamps = get_all_avi_files(directory_path="C:/Users/MSAD/github/nnspike/storage/20250902/videos/")
avi_files_with_timestamps

[('C:/Users/MSAD/github/nnspike/storage/20250902/videos/20250820163700_picamera.avi',
  '20250820163700'),
 ('C:/Users/MSAD/github/nnspike/storage/20250902/videos/20250820164500_picamera.avi',
  '20250820164500'),
 ('C:/Users/MSAD/github/nnspike/storage/20250902/videos/20250820171957_picamera.avi',
  '20250820171957'),
 ('C:/Users/MSAD/github/nnspike/storage/20250902/videos/20250820172417_picamera.avi',
  '20250820172417'),
 ('C:/Users/MSAD/github/nnspike/storage/20250902/videos/20250820172631_picamera.avi',
  '20250820172631'),
 ('C:/Users/MSAD/github/nnspike/storage/20250902/videos/20250820173023_picamera.avi',
  '20250820173023'),
 ('C:/Users/MSAD/github/nnspike/storage/20250902/videos/20250820173414_picamera.avi',
  '20250820173414'),
 ('C:/Users/MSAD/github/nnspike/storage/20250902/videos/20250820173808_picamera.avi',
  '20250820173808'),
 ('C:/Users/MSAD/github/nnspike/storage/20250902/videos/20250820173842_picamera.avi',
  '20250820173842'),
 ('C:/Users/MSAD/github/nnspike/stora

In [4]:
    # Extract frames from all AVI files
if avi_files_with_timestamps:
    print("\nStarting frame extraction...")
    output_dirs_with_timestamps = extract_frames_from_avi_files(avi_files_with_timestamps, base_output_dir="C:/Users/MSAD/github/nnspike/storage/20250902/frames/")
    print("\nFrame extraction completed!")
    print(f"Successfully created {len(output_dirs_with_timestamps)} output directories:")
    for output_dir, timestamp in output_dirs_with_timestamps:
        print(f"  - {output_dir} (timestamp: {timestamp})")
else:
    print("No AVI files found to process.")
    output_dirs_with_timestamps = []
    
output_dirs_with_timestamps


Starting frame extraction...
Extracting frames from 20250820163700_picamera.avi to C:/Users/MSAD/github/nnspike/storage/20250902/frames/20250820163700/
Frames extracted to: C:\Users\MSAD\github\nnspike\storage\20250902\frames\20250820163700
✓ Successfully extracted frames to 20250820163700/
Extracting frames from 20250820164500_picamera.avi to C:/Users/MSAD/github/nnspike/storage/20250902/frames/20250820164500/
Frames extracted to: C:\Users\MSAD\github\nnspike\storage\20250902\frames\20250820164500
✓ Successfully extracted frames to 20250820164500/
Extracting frames from 20250820171957_picamera.avi to C:/Users/MSAD/github/nnspike/storage/20250902/frames/20250820171957/
Frames extracted to: C:\Users\MSAD\github\nnspike\storage\20250902\frames\20250820171957
✓ Successfully extracted frames to 20250820171957/
Extracting frames from 20250820172417_picamera.avi to C:/Users/MSAD/github/nnspike/storage/20250902/frames/20250820172417/
Frames extracted to: C:\Users\MSAD\github\nnspike\storage\

[('C:/Users/MSAD/github/nnspike/storage/20250902/frames/20250820163700',
  '20250820163700'),
 ('C:/Users/MSAD/github/nnspike/storage/20250902/frames/20250820164500',
  '20250820164500'),
 ('C:/Users/MSAD/github/nnspike/storage/20250902/frames/20250820171957',
  '20250820171957'),
 ('C:/Users/MSAD/github/nnspike/storage/20250902/frames/20250820172417',
  '20250820172417'),
 ('C:/Users/MSAD/github/nnspike/storage/20250902/frames/20250820172631',
  '20250820172631'),
 ('C:/Users/MSAD/github/nnspike/storage/20250902/frames/20250820173023',
  '20250820173023'),
 ('C:/Users/MSAD/github/nnspike/storage/20250902/frames/20250820173414',
  '20250820173414'),
 ('C:/Users/MSAD/github/nnspike/storage/20250902/frames/20250820173808',
  '20250820173808'),
 ('C:/Users/MSAD/github/nnspike/storage/20250902/frames/20250820173842',
  '20250820173842'),
 ('C:/Users/MSAD/github/nnspike/storage/20250902/frames/20250820185312',
  '20250820185312'),
 ('C:/Users/MSAD/github/nnspike/storage/20250902/frames/2025

In [6]:
import os
for output_dir, timestamp in output_dirs_with_timestamps:
    print(f"Output directory: {output_dir} (timestamp: {timestamp})")
    
    course = "right"
    label_df = create_label_dataframe(output_dir +"/*", course)
    label_df = sort_by_frames_number(label_df)
    label_df = label_dataset_by_opencv(label_df, ROI_CNN, 80)
    
    sensor_path = f"C:/Users/MSAD/github/nnspike/storage/20250902/sensor_data/{timestamp}_sensor_log.csv"
    if not os.path.exists(sensor_path):
        print(f"Warning: Sensor log not found for timestamp {timestamp}, skipping.")
        continue
    
    status_df = pd.read_csv(sensor_path)
    df = set_spike_status(label_df, status_df)
    
    # Export to a csv file
    df.to_csv(f"C:/Users/MSAD/github/nnspike/storage/20250902/labels/{timestamp}_label.csv", index=False)

Output directory: C:/Users/MSAD/github/nnspike/storage/20250902/frames/20250820163700 (timestamp: 20250820163700)


Processing: 100%|██████████| 1559/1559 [00:24<00:00, 62.54it/s]



Output directory: C:/Users/MSAD/github/nnspike/storage/20250902/frames/20250820164500 (timestamp: 20250820164500)


Processing: 100%|██████████| 848/848 [00:23<00:00, 36.71it/s]



Output directory: C:/Users/MSAD/github/nnspike/storage/20250902/frames/20250820171957 (timestamp: 20250820171957)


...

Processing: 100%|██████████| 310/310 [00:08<00:00, 35.21it/s]



Output directory: C:/Users/MSAD/github/nnspike/storage/20250902/frames/20250902182833 (timestamp: 20250902182833)


Processing: 100%|██████████| 1778/1778 [00:56<00:00, 31.37it/s]



Output directory: C:/Users/MSAD/github/nnspike/storage/20250902/frames/20250902183218 (timestamp: 20250902183218)


Processing: 100%|██████████| 1803/1803 [00:54<00:00, 33.24it/s]

